In [1]:
import torch

from mpc import mpc
from mpc.mpc import QuadCost, LinDx, GradMethods
from mpc.env_dx import cartpole

import numpy as np
import numpy.random as npr

import matplotlib.pyplot as plt

import os
import io
import base64
import tempfile
from IPython.display import HTML

from tqdm import tqdm

%matplotlib inline

In [2]:
dx = cartpole.CartpoleDx()

n_batch, T, mpc_T = 8, 100, 25

# 用于生成均匀分布
def uniform(shape, low, high):
    r = high-low
    return torch.rand(shape)*r+low

torch.manual_seed(0)
# 初始状态
th = uniform(n_batch, -2*np.pi, 2*np.pi)
thdot = uniform(n_batch, -.5, .5)
x = uniform(n_batch, -0.5, 0.5)
xdot = uniform(n_batch, -0.5, 0.5)
xinit = torch.stack((x, xdot, torch.cos(th), torch.sin(th), thdot), dim=1)
print(xinit)

x = xinit
u_init = None

q, p = dx.get_true_obj()
print(q)
print(p)

# 每个batch和每个timestamp的运行损失都是一样的，这里就是在构造这样的Q和p
Q = torch.diag(q).unsqueeze(0).unsqueeze(0).repeat(
    mpc_T, n_batch, 1, 1
)
p = p.unsqueeze(0).repeat(mpc_T, n_batch, 1)

t_dir = "D:/Docs/code_lib/graduation_test/cartpole_pic"

action_history = []
for t in tqdm(range(T)):
    nominal_states, nominal_actions, nominal_objs = mpc.MPC(
        dx.n_state, dx.n_ctrl, mpc_T,
        u_init=u_init,                                  # u_init的作用: warm-start，表示对控制序列的初始猜测 (用上一个时刻的预测结果)，可以加速收敛
        # u_lower=dx.lower, u_upper=dx.upper,
        lqr_iter=50,
        verbose=0,
        exit_unconverged=False,
        detach_unconverged=False,
        linesearch_decay=dx.linesearch_decay,
        max_linesearch_iter=dx.max_linesearch_iter,
        grad_method=GradMethods.AUTO_DIFF,
        eps=1e-2,
    )(x, QuadCost(Q, p), dx)
    
    print(nominal_states)
    print(nominal_actions)
    print(nominal_objs)
    break
    
    next_action = nominal_actions[0]
    action_history.append(next_action)
    u_init = torch.cat((nominal_actions[1:], torch.zeros(1, n_batch, dx.n_ctrl)), dim=0) # u_init shape (mpc_T, batch_size, 1)
    u_init[-2] = u_init[-3] # 不知道为什么这样写，猜想应该是u[-1] = u[-2]才对

    x = dx(x, next_action) # 往前走一步

    # 以下都是用来可视化的
    n_col = 4
    n_row = n_batch // n_col
    fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
    axs = axs.reshape(-1)
    for i in range(n_batch):
        dx.get_frame(x[i], ax=axs[i])
        axs[i].get_xaxis().set_visible(False)
        axs[i].get_yaxis().set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(t_dir, 'frame_{:03d}.png'.format(t)))
    plt.close(fig)
    
action_history = torch.stack(action_history).detach()[:,:,0]

tensor([[ 0.1977, -0.0806,  0.9989, -0.0470, -0.0444],
        [ 0.3000,  0.0529, -0.9739, -0.2270,  0.1323],
        [-0.3390,  0.4527,  0.4430,  0.8965, -0.1511],
        [-0.2177, -0.4638, -0.0882,  0.9961, -0.0983],
        [ 0.1816, -0.3148, -0.7508, -0.6606, -0.4777],
        [ 0.4152, -0.1266, -0.1138,  0.9935, -0.3311],
        [-0.1029, -0.1949,  0.9923, -0.1242, -0.2061],
        [ 0.3742,  0.4320,  0.2662, -0.9639,  0.0185]])
tensor([0.1000, 0.1000, 1.0000, 1.0000, 0.1000, 0.0010])
tensor([-0., -0., -1., -0., -0.,  0.])


  0%|          | 0/100 [00:04<?, ?it/s]

tensor([[[ 1.9767e-01, -8.0592e-02,  9.9889e-01, -4.7024e-02, -4.4372e-02],
         [ 3.0001e-01,  5.2907e-02, -9.7390e-01, -2.2699e-01,  1.3231e-01],
         [-3.3897e-01,  4.5274e-01,  4.4301e-01,  8.9652e-01, -1.5111e-01],
         [-2.1773e-01, -4.6384e-01, -8.8233e-02,  9.9610e-01, -9.8283e-02],
         [ 1.8161e-01, -3.1477e-01, -7.5075e-01, -6.6058e-01, -4.7767e-01],
         [ 4.1519e-01, -1.2658e-01, -1.1384e-01,  9.9350e-01, -3.3114e-01],
         [-1.0290e-01, -1.9490e-01,  9.9226e-01, -1.2417e-01, -2.0611e-01],
         [ 3.7416e-01,  4.3200e-01,  2.6623e-01, -9.6391e-01,  1.8522e-02]],

        [[ 1.9364e-01, -2.6351e-01,  9.9879e-01, -4.9240e-02,  1.9514e-01],
         [ 3.0266e-01,  1.9644e-01, -9.7238e-01, -2.3342e-01,  1.7514e-01],
         [-3.1633e-01,  2.8714e-01,  4.4977e-01,  8.9314e-01,  6.1787e-01],
         [-2.4092e-01, -2.1578e-01, -8.3337e-02,  9.9652e-01,  6.6668e-01],
         [ 1.6587e-01, -4.1136e-01, -7.6631e-01, -6.4247e-01, -1.0720e+00],
         [

RuntimeError: stack expects a non-empty TensorList

In [11]:
# Plot actions
for t in tqdm(range(T)):
    fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
    axs = axs.reshape(-1)
    for i in range(n_batch):
        axs[i].plot(action_history[:,i], color='k')
        axs[i].set_ylim(-15, 15)
        axs[i].axvline(t, color='k', ls='--', linewidth=4)
        axs[i].get_xaxis().set_visible(False)
        axs[i].get_yaxis().set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(t_dir, 'actions_{:03d}.png'.format(t)))
    plt.close(fig)
    
    f1 = os.path.join(t_dir, 'frame_{:03d}.png'.format(t))
    f2 = os.path.join(t_dir, 'actions_{:03d}.png'.format(t))
    f_out = os.path.join(t_dir, '{:03d}.png'.format(t))
    os.system(f'convert {f1} {f2} +append -resize 1200x {f_out}')

100%|██████████| 100/100 [00:14<00:00,  7.13it/s]


In [5]:
vid_fname = 'cartpole.mp4'

if os.path.exists(vid_fname):
    os.remove(vid_fname)
    
t_dir = 'D:/Docs/code_lib/graduation_test/cartpole_pic'

cmd = 'ffmpeg -r 16 -f image2 -i {}/frame_%03d.png -vcodec libx264 -crf 25 -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -pix_fmt yuv420p {}'.format(
    t_dir, vid_fname
)
print(cmd)
os.system(cmd)
print('Saving video to: {}'.format(vid_fname))

ffmpeg -r 16 -f image2 -i D:/Docs/code_lib/graduation_test/cartpole_pic/frame_%03d.png -vcodec libx264 -crf 25 -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -pix_fmt yuv420p cartpole.mp4
Saving video to: cartpole.mp4


In [8]:
video = io.open(vid_fname, 'r+b').read()
encoded = base64.b64encode(video)
HTML(data='''<video alt="test" controls>
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii')))

FileNotFoundError: [Errno 2] No such file or directory: 'cartpole2.mp4'